# 00 · Full pipeline in one Kaggle session
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Kaggle notebooks do not share `/kaggle/working`, so on Kaggle the simplest route is to run everything here in one
session (GPU T4, Internet ON). Notebooks 01–05 contain the same stages with more inspection cells.

Inputs: the Kaggle dataset `prasukjain07/preprocessed-alzheimers-disease-mri-dataset` (or it is downloaded with
`kagglehub`) and the project code (`dementia_mri_project.zip` uploaded as a Kaggle dataset).
Approximate T4 time: audit + split 3–6 min, EfficientNet-B0 ~0.5–1 h, EfficientNetV2-S ~1.5–3 h, ablation +2 runs.
Set `CONFIG_OVERRIDES = {"ablation": {"enabled": False}}` to skip the extra ablation trainings.

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
from src.run_pipeline import stage_audit, stage_split, stage_train, stage_evaluate, stage_gradcam, stage_report, stage_package
AUDIT, SUMMARY = stage_audit(cfg, paths)            # STEPS 1-11

In [ ]:
META, SPLIT = stage_split(cfg, paths)               # STEPS 12-16 (stops if leakage is detected)
show(paths["results"] / "split_class_distribution.png")

In [ ]:
RUNS = stage_train(cfg, paths)                      # STEPS 17-19 (baseline, then primary)

In [ ]:
RES = stage_evaluate(cfg, paths, ablation=cfg["ablation"]["enabled"])   # STEPS 20-25, 30-32
show(paths["results"] / "normalized_confusion_matrix.png", 520)

In [ ]:
CAM = stage_gradcam(cfg, paths)                     # STEPS 26-29

In [ ]:
stage_report(cfg, paths); stage_package(cfg, paths)   # STEPS 33-35
show_md(paths["root"] / "FINAL_REPORT.md")